# 03 — Table 1: Baseline characteristics by outcome

Classic **Table 1** for the ICU cohort in `cohort.csv` (from `01_cohort_extraction.ipynb`), stratified by delirium label (`label`: 0 = no CAM-ICU–defined delirium during stay, 1 = delirium).

- Continuous variables: mean ± standard deviation; **Welch’s t-test** (two-sided).
- Categorical variables: n (% of column); **Pearson χ²** (or **Fisher’s exact** for 2×2 tables when expected counts are small).

Adjust `OUTPUT_DIR` if your `cohort.csv` lives elsewhere.

In [3]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

OUTPUT_DIR = Path("/users/syang195/1595-final")
RESULTS_DIR = OUTPUT_DIR / "results" / "table1"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.max_colwidth = 80
np.random.seed(42)

In [4]:
cohort = pd.read_csv(
    OUTPUT_DIR / "cohort.csv",
    parse_dates=["intime", "outtime", "first_delirium_time"],
)

assert "label" in cohort.columns, "cohort.csv must contain a 'label' column"

cohort["label"] = cohort["label"].astype(int)
g0 = cohort["label"] == 0
g1 = cohort["label"] == 1
n0, n1 = int(g0.sum()), int(g1.sum())
n = len(cohort)

print(f"Total stays: {n:,}  |  No delirium: {n0:,}  |  Delirium: {n1:,}")
cohort.head(2)

Total stays: 60,132  |  No delirium: 45,511  |  Delirium: 14,621


,subject_id,hadm_id,stay_id,intime,outtime,los_hours,anchor_age,gender,race,insurance,marital_status,first_careunit,label,first_delirium_time
0,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,93.438056,86,F,WHITE,Medicare,WIDOWED,Medical Intensive Care Unit (MICU),0,NaT
1,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,26.832778,55,F,WHITE,Private,MARRIED,Surgical Intensive Care Unit (SICU),0,NaT


## Build Table 1

In [5]:
def _p_fmt(p: float) -> str:
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def _mean_sd(series: pd.Series) -> str:
    x = series.dropna().astype(float)
    if len(x) == 0:
        return "—"
    return f"{x.mean():.1f} ± {x.std(ddof=1):.1f}"


def _pct_str(k: int, denom: int) -> str:
    if denom == 0:
        return "—"
    return f"{k} ({100.0 * k / denom:.1f})"


def p_continuous_ttest(a: pd.Series, b: pd.Series) -> float:
    a = a.dropna().astype(float)
    b = b.dropna().astype(float)
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    _, p = stats.ttest_ind(a, b, equal_var=False)
    return float(p)


def p_categorical(cat: pd.Series, label: pd.Series) -> float:
    df = pd.DataFrame({"c": cat.astype(str), "l": label.astype(int)})
    df = df.dropna(subset=["c"])
    tab = pd.crosstab(df["c"], df["l"])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return float("nan")
    chi2, p, dof, expected = stats.chi2_contingency(tab)
    if expected.min() < 5 and tab.shape == (2, 2):
        _, p = stats.fisher_exact(tab.values)
    return float(p)


rows: list[dict[str, str]] = []

col_stable = f"No delirium (n={n0:,})"
col_delir = f"Delirium (n={n1:,})"

# --- Sample size row ---
rows.append(
    {
        "Characteristic": "ICU stays, n",
        col_stable: f"{n0:,}",
        col_delir: f"{n1:,}",
        "P-value": "",
    }
)

# --- Continuous (mean ± SD) ---
continuous = [
    ("Age at anchor (years)", "anchor_age"),
    ("ICU length of stay (hours)", "los_hours"),
]
for label, col in continuous:
    if col not in cohort.columns:
        continue
    p = p_continuous_ttest(cohort.loc[g0, col], cohort.loc[g1, col])
    rows.append(
        {
            "Characteristic": f"  {label}, mean ± SD",
            col_stable: _mean_sd(cohort.loc[g0, col]),
            col_delir: _mean_sd(cohort.loc[g1, col]),
            "P-value": _p_fmt(p) if np.isfinite(p) else "—",
        }
    )

# --- Categorical ---
categorical = [
    ("Sex", "gender"),
    ("Race", "race"),
    ("Insurance", "insurance"),
    ("Marital status", "marital_status"),
    ("First ICU unit", "first_careunit"),
]

for block_label, col in categorical:
    if col not in cohort.columns:
        continue
    s = cohort[col].fillna("Missing").astype(str).str.strip()
    s = s.replace({"": "Missing"})
    p = p_categorical(s, cohort["label"])
    p_str = _p_fmt(p) if np.isfinite(p) else "—"

    levels = sorted(s.unique(), key=lambda x: (-(s == x).sum(), x))
    for i, lev in enumerate(levels):
        k0 = int((s[g0] == lev).sum())
        k1 = int((s[g1] == lev).sum())
        rows.append(
            {
                "Characteristic": f"{block_label}: {lev}",
                col_stable: _pct_str(k0, n0),
                col_delir: _pct_str(k1, n1),
                "P-value": p_str if i == 0 else "",
            }
        )

table1 = pd.DataFrame(rows)
table1

,Characteristic,"No delirium (n=45,511)","Delirium (n=14,621)",P-value
0,"ICU stays, n","45,511","14,621",
1,"Age at anchor (years), mean ± SD",63.5 ± 16.3,65.5 ± 15.4,<0.001
2,"ICU length of stay (hours), mean ± SD",77.8 ± 91.2,182.6 ± 203.3,<0.001
3,Sex: M,26506 (58.2),8676 (59.3),0.020
4,Sex: F,19005 (41.8),5945 (40.7),
...,...,...,...,...
61,First ICU unit: Medicine,2 (0.0),8 (0.1),
62,First ICU unit: Surgery/Trauma,0 (0.0),3 (0.0),
63,First ICU unit: Med/Surg,0 (0.0),1 (0.0),
64,First ICU unit: Medicine/Cardiology Intermediate,0 (0.0),1 (0.0),


In [6]:
out_csv = RESULTS_DIR / "table1_baseline_by_label.csv"
table1.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")

# Publication-style plain text (optional paste into manuscript)
print()
print(table1.to_string(index=False))

Saved: /users/syang195/1595-final/results/table1/table1_baseline_by_label.csv

                                                  Characteristic No delirium (n=45,511) Delirium (n=14,621) P-value
                                                    ICU stays, n                 45,511              14,621        
                                Age at anchor (years), mean ± SD            63.5 ± 16.3         65.5 ± 15.4  <0.001
                           ICU length of stay (hours), mean ± SD            77.8 ± 91.2       182.6 ± 203.3  <0.001
                                                          Sex: M           26506 (58.2)         8676 (59.3)   0.020
                                                          Sex: F           19005 (41.8)         5945 (40.7)        
                                                     Race: WHITE           29310 (64.4)         8252 (56.4)  <0.001
                                    Race: BLACK/AFRICAN AMERICAN             4003 (8.8)         1462 (10.0)  